# 04 — Train adv-NO on dynamic_dt export

Trains on data from [`01_export_dynamic_dt.ipynb`](01_export_dynamic_dt.ipynb).

| Knob | Description |
|------|-------------|
| `RUN_ALL_DT` | **`True`** (default) → train all 8 dt in parallel on **cuda:0–7** |
| `DT_DAYS` | Single dt when `RUN_ALL_DT=False` |
| `EXPORT_KEY` | `magicwin_reference` or `smallfixwin` (both uv, iD=2) |
| `EPOCHS` | **200** |
| `OPTIMIZER` | **adamw** (`WEIGHT_DECAY=1e-3`) |
| `VAL_SPLIT_MODE` | **`export`** — use train/val files from 01 (no merge/re-split) |

**Note:** Export (01) runs **sequentially on CPU**; train/rollout use **GPUs in parallel**.

**Sequence:** 5× conditioning (`t-4…t`) → predict `t+1`. Train on u,v increments; rollout eval on **speed**.

**dt → GPU:** 3→0, 5→1, 7→2, 10→3, 12→4, 15→5, 17→6, 20→7

Run [`05_evaluate_rollout_dynamic_dt.ipynb`](05_evaluate_rollout_dynamic_dt.ipynb) after training.

In [ ]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "dynamic_dt" else (
    NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
)
DT_DIR = MW_DIR / "dynamic_dt"
ROOT = MW_DIR.parent
for _p in (ROOT, MW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import dynamic_dt.multi_dt_runner as _mdr
import dynamic_dt.train_runner as _tr
importlib.reload(_mdr)
importlib.reload(_tr)
from dynamic_dt.device_plan import format_dt_device_plan
from dynamic_dt.dt_config import dt_hours
from dynamic_dt.window_catalog import get_window_spec, primary_export_component
from dynamic_dt.window_paths import train_prefix, val_prefix
from dynamic_dt.train_runner import build_train_config, prepare_train_job, train_one_dt
from dynamic_dt.multi_dt_runner import launch_train_all, print_job_summary
from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training

MODELS_DIR = DT_DIR / "Models" / "adv_no"
LOG_DIR = DT_DIR / "logs"
for _d in (DT_DIR / "data", MODELS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ============ USER KNOBS ============
EXPORT_KEY = "magicwin_reference"   # or "smallfixwin"
RUN_ALL_DT = True                   # all 8 dt on GPUs 0–7
DT_DAYS = 15                        # only when RUN_ALL_DT=False

MAX_SAMPLES = None
EPOCHS = 200
OPTIMIZER = "adamw"
WEIGHT_DECAY = 1e-3
VAL_SPLIT = 0.3                  # ignored when VAL_SPLIT_MODE="export"
VAL_SPLIT_MODE = "export"        # use train/val files from 01_export_dynamic_dt.ipynb
VAL_SPLIT_SEED = 42
VAL_RECONSTRUCTION_METRIC = "l1"
DETACHED = True
RESUME = True
# ====================================

WINDOW = get_window_spec(EXPORT_KEY)
EXPORT_COMPONENT = primary_export_component(EXPORT_KEY)
train_kw = dict(
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    max_samples=MAX_SAMPLES,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
)

print(f"EXPORT_KEY={EXPORT_KEY} ({WINDOW['label']}) | RUN_ALL_DT={RUN_ALL_DT}")
print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}, epochs={EPOCHS}")
print(f"val split: export (train/val prefixes from 01_export)")
print(format_dt_device_plan(None if RUN_ALL_DT else [DT_DAYS]))

In [ ]:
if RUN_ALL_DT:
    jobs = launch_train_all(
        dt_dir=DT_DIR,
        root=ROOT,
        export_key=EXPORT_KEY,
        log_dir=LOG_DIR,
        detached=DETACHED,
        resume=RESUME,
        **train_kw,
    )
    print_job_summary(jobs, label="Train")
else:
    import numpy as np
    DT = dt_hours(DT_DAYS)
    data_prefix = train_prefix(DT_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt_hours=DT)
    val_prefix_path = val_prefix(DT_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt_hours=DT)
    for label, prefix in (("train", data_prefix), ("val", val_prefix_path)):
        p = Path(f"{prefix}_sims.npy")
        if not p.exists():
            raise FileNotFoundError(f"Missing {label} export: {p}\nRun 01_export_dynamic_dt.ipynb")
        print(f"{label}: {np.load(str(p), mmap_mode='r').shape}")

    if DETACHED:
        launched = train_one_dt(
            dt_dir=DT_DIR, root=ROOT, export_key=EXPORT_KEY, dt_days=DT_DAYS,
            log_dir=LOG_DIR, detached=True, resume=RESUME, **train_kw,
        )
        print(f"Started PID {launched['pid']} | tail -f {launched['log_path']}")
    else:
        job = prepare_train_job(dt_dir=DT_DIR, export_key=EXPORT_KEY, dt_days=DT_DAYS, **train_kw)
        config = AdvNOTrainConfig.from_dict(json.loads(Path(job["config_path"]).read_text()))
        run_adv_no_training(config)